# GAPFA Notebook 12 v2 — Statistical Analysis and Figures

This notebook consolidates the frozen results from:

- **Notebook 10:** DINOv2-Small confirmatory evaluation.
- **Notebook 11:** ResNet-50 (`IMAGENET1K_V2`) backbone robustness.

It performs protocol audit, `mean ± std`, paired effects, paired bootstrap 95% CIs, paired permutation tests, global Holm correction, win/tie/loss analysis, cross-backbone consistency, and paper-ready figures.

**Runtime:** CPU / No accelerator. **Internet:** Off.

Expected frozen methods:
`LR-Real`, `PFG-UnweightedLR`, `PFG-NW-F1Only`, `CA-PFG-MO`.

Expected protocol: **60 unique tasks per backbone**.


**v2 loader fix:** robust prefixed/suffixed metric-column normalization and strict exclusion of smoke results from full-protocol source selection.

> **Critical v3 correction (17 November 2026):** Explicitly map `test_macro_f1`, `test_nll`, `test_ece`, and `test_brier` to analysis metrics. The original `macro_f1`/`nll`/`ece`/`brier` source fields are validation values and must never be reported as held-out test results. Original selected classifiers, tasks, and frozen seeds remain unchanged. The corrected notebook fails if the required test columns are absent.


## Input

Recommended Kaggle inputs:

```text
/kaggle/input/
├── gapfa10-dinov2-confirmatory/
│   └── ... result CSV(s) or ZIP from Notebook 10 ...
└── gapfa11-resnet50-robustness/
    └── ... result CSV(s) or ZIP from Notebook 11 ...
```

The notebook recursively scans ordinary CSV files and CSV files inside ZIP archives.

Each backbone result table must provide columns equivalent to:

`dataset`, `shot`/`k_shot`, `seed`, `method`, `macro_f1`, `nll`, `ece`

`brier` is optional.

If automatic source detection is ambiguous, set the two manual paths in Cell 2. ZIP members use the form:

```python
"/kaggle/input/source/results.zip::folder/all_results.csv"
```


In [ ]:
# Cell 1 — Imports and reproducibility
from __future__ import annotations
import json, re, zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

RANDOM_SEED = 20260706
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Seed:", RANDOM_SEED)


In [ ]:
# Cell 2 — Configuration
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/GAPFA12_v3_TEST_only_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Leave None for automatic discovery.
DINO_RESULT_PATH: Optional[str] = None
RESNET_RESULT_PATH: Optional[str] = None

EXPECTED_TASKS_PER_BACKBONE = 60
EXPECTED_METHODS = [
    "LR-Real",
    "PFG-UnweightedLR",
    "PFG-NW-F1Only",
    "CA-PFG-MO",
]
METHOD_ORDER = EXPECTED_METHODS.copy()

# Component-wise frozen contrasts: reference -> challenger
PRIMARY_CONTRASTS = {
    "R1_PFG": ("LR-Real", "PFG-UnweightedLR"),
    "R2_NormalizedBudget": ("PFG-UnweightedLR", "PFG-NW-F1Only"),
    "R3_CalibrationAwareMO": ("PFG-NW-F1Only", "CA-PFG-MO"),
}

METRIC_DIRECTIONS = {
    "macro_f1": "higher",
    "nll": "lower",
    "ece": "lower",
    "brier": "lower",
}
PRIMARY_METRICS = ["macro_f1", "nll", "ece"]
OPTIONAL_METRICS = ["brier"]

# Frozen confirmatory family for multiplicity control:
# R2 and R3 × Macro-F1/NLL/ECE × 2 backbones = 12 hypotheses.
HOLM_CONFIRMATORY_CONTRASTS = ["R2_NormalizedBudget", "R3_CalibrationAwareMO"]

BOOTSTRAP_REPS = 20_000
PERMUTATION_REPS = 100_000
ALPHA = 0.05
TIE_TOL = 1e-12

print("OUTPUT_DIR:", OUTPUT_DIR)


## 1. Discover and normalize the two frozen result tables

In [ ]:
# Cell 3 — Robust schema aliases and generic loaders
COLUMN_ALIASES = {
    "dataset": ["dataset", "dataset_name", "data_name", "corpus"],
    "shot": ["shot", "shots", "k", "k_shot", "kshot", "n_shot", "nshot",
             "num_shots", "support_shot", "support_size"],
    "seed": ["seed", "random_seed", "run_seed", "trial_seed"],
    "method": ["method", "model", "approach", "variant", "method_name"],
    "macro_f1": ["macro_f1", "f1_macro", "macrof1", "macro-f1", "f1"],
    "nll": ["nll", "log_loss", "negative_log_likelihood",
            "neg_log_likelihood", "cross_entropy"],
    "ece": ["ece", "expected_calibration_error", "calibration_error"],
    "brier": ["brier", "brier_score", "multiclass_brier", "brier_loss"],
    "backbone": ["backbone", "encoder", "feature_extractor"],
}

def canon(x):
    x = str(x).strip().lower()
    x = re.sub(r"[\s\-/]+", "_", x)
    x = re.sub(r"[^a-z0-9_]+", "", x)
    return re.sub(r"_+", "_", x).strip("_")

ALIAS_LOOKUP = {}
for target, aliases in COLUMN_ALIASES.items():
    for a in aliases + [target]:
        ALIAS_LOOKUP[canon(a)] = target

def metric_target_from_extended_name(key: str):
    """
    Recognize task-level metric columns with prefixes/suffixes, e.g.:
    test_macro_f1, macro_f1_test, test_f1_macro,
    test_nll, nll_test, test_ece, ece_test,
    test_brier, brier_score_test.
    """
    k = canon(key)

    if re.search(r"(^|_)(macro_f1|f1_macro|macrof1)($|_)", k):
        return "macro_f1"
    if re.search(r"(^|_)(nll|log_loss|negative_log_likelihood|neg_log_likelihood)($|_)", k):
        return "nll"
    if re.search(r"(^|_)(ece|expected_calibration_error|calibration_error)($|_)", k):
        return "ece"
    if re.search(r"(^|_)(brier|brier_score|multiclass_brier|brier_loss)($|_)", k):
        return "brier"
    return None

def normalize_columns(df):
    """
    Two-pass normalization:
      1) exact aliases;
      2) robust metric-name detection with prefixes/suffixes.
    Exact aliases always have priority.
    """
    rename, used = {}, set()

    # Pass 1: exact aliases
    for c in df.columns:
        t = ALIAS_LOOKUP.get(canon(c))
        if t is not None and t not in used:
            rename[c] = t
            used.add(t)

    # Pass 2: prefixed/suffixed metric columns
    for c in df.columns:
        if c in rename:
            continue
        t = metric_target_from_extended_name(c)
        if t is not None and t not in used:
            rename[c] = t
            used.add(t)

    out = df.rename(columns=rename).copy()
    return out

def read_spec(spec: str):
    if "::" in spec:
        zp, member = spec.split("::", 1)
        with zipfile.ZipFile(zp, "r") as zf, zf.open(member) as fh:
            return pd.read_csv(fh)
    return pd.read_csv(spec)

def iter_csv_specs(root: Path) -> Iterable[str]:
    for p in sorted(root.rglob("*.csv")):
        yield str(p)
    for z in sorted(root.rglob("*.zip")):
        try:
            with zipfile.ZipFile(z, "r") as zf:
                for member in sorted(zf.namelist()):
                    if member.lower().endswith(".csv") and not member.endswith("/"):
                        yield f"{z}::{member}"
        except zipfile.BadZipFile:
            print("WARNING bad ZIP:", z)

def norm_method(x):
    s = str(x).strip()
    k = re.sub(r"[^a-z0-9]+", "", s.lower())
    mp = {
        "lrreal": "LR-Real",
        "reallr": "LR-Real",
        "pfgunweightedlr": "PFG-UnweightedLR",
        "pfgnwf1only": "PFG-NW-F1Only",
        "capfgmo": "CA-PFG-MO",
    }
    return mp.get(k, s)

def norm_shot(x):
    if pd.isna(x):
        return x
    s = str(x).strip()
    m = re.search(r"\d+", s)
    return int(m.group()) if m else s

def normalize_result(df, backbone, source_spec=None):
    raw_columns = list(df.columns)
    out = normalize_columns(df)

    # CRITICAL v3: demand explicit held-out test metric columns from NB10/NB11.
    # Generic macro_f1/nll/ece/brier in the source refer to VALIDATION,
    # used for model selection; they cannot be reported as test metrics.
    metrics = ("macro_f1", "nll", "ece", "brier")
    required_test = [f"test_{m}" for m in metrics]
    missing_test = [col for col in required_test if col not in out.columns]
    if missing_test:
        raise ValueError(
            f"Missing explicit held-out TEST metrics in {source_spec}: {missing_test}. "
            "Do not substitute validation metrics or smoke results."
        )
    for m in metrics:
        if m in out.columns:
            out[f"validation_{m}"] = out[m].copy()
        out[m] = out[f"test_{m}"].copy()

    required = {"dataset", "shot", "seed", "method", "macro_f1", "nll", "ece"}
    missing = required - set(out.columns)

    if missing:
        print("\nSOURCE:", source_spec if source_spec is not None else "<dataframe>")
        print("RAW COLUMNS:")
        print(raw_columns)
        print("\nNORMALIZED COLUMNS:")
        print(list(out.columns))
        raise ValueError(
            f"Missing required columns after robust normalization: {sorted(missing)}"
        )

    out["dataset"] = out["dataset"].astype(str).str.strip()
    out["shot"] = out["shot"].map(norm_shot)
    out["seed"] = pd.to_numeric(out["seed"], errors="raise").astype(int)
    out["method"] = out["method"].map(norm_method)

    for m in ["macro_f1", "nll", "ece", "brier"]:
        if m in out:
            out[m] = pd.to_numeric(out[m], errors="coerce")

    out["backbone"] = backbone
    return out

print("Robust loader ready.")


In [ ]:
# Cell 4 — Candidate audit and safe full-run source selection
@dataclass
class Candidate:
    spec: str
    n_rows: int
    n_tasks: int
    n_expected_methods: int
    schema_ok: bool
    full_protocol_ok: bool
    is_smoke: bool
    path_hint: str
    score: float

def path_hint(spec):
    s = spec.lower()
    if any(k in s for k in ["dinov2", "dino_v2", "notebook10", "nb10", "gapfa10"]):
        return "DINOv2-Small"
    if any(k in s for k in ["resnet50", "resnet-50", "notebook11", "nb11", "gapfa11"]):
        return "ResNet-50"
    return "Unknown"

def inspect(spec):
    try:
        raw = read_spec(spec)
    except Exception as e:
        print("SKIP:", spec, repr(e))
        return None

    n = normalize_columns(raw)
    req = {"dataset", "shot", "seed", "method", "macro_f1", "nll", "ece"}
    schema_ok = req.issubset(n.columns) and {"test_macro_f1", "test_nll", "test_ece", "test_brier"}.issubset(n.columns)

    nt, nm = 0, 0
    if schema_ok:
        nt = n[["dataset", "shot", "seed"]].drop_duplicates().shape[0]
        nm = len(set(n["method"].map(norm_method)) & set(EXPECTED_METHODS))

    smoke = "smoke" in spec.lower()
    full_ok = (
        schema_ok
        and nt == EXPECTED_TASKS_PER_BACKBONE
        and nm == len(EXPECTED_METHODS)
        and not smoke
    )

    hint = path_hint(spec)

    # Full 60-task protocol dominates all other scoring.
    score = (
        (10000 if full_ok else 0)
        + (1000 if schema_ok else 0)
        + 100 * nm
        + min(nt, EXPECTED_TASKS_PER_BACKBONE)
        - 0.25 * abs(nt - EXPECTED_TASKS_PER_BACKBONE)
        + (25 if hint != "Unknown" else 0)
        - (5000 if smoke else 0)
    )

    return Candidate(
        spec=spec,
        n_rows=len(raw),
        n_tasks=nt,
        n_expected_methods=nm,
        schema_ok=schema_ok,
        full_protocol_ok=full_ok,
        is_smoke=smoke,
        path_hint=hint,
        score=score,
    )

specs = list(iter_csv_specs(INPUT_ROOT))
cands = [c for s in specs if (c := inspect(s)) is not None]
candidate_df = pd.DataFrame([c.__dict__ for c in cands])

if candidate_df.empty:
    raise RuntimeError("No readable CSV candidate found under /kaggle/input.")

candidate_df = candidate_df.sort_values(
    ["full_protocol_ok", "schema_ok", "score", "n_tasks", "n_rows"],
    ascending=[False, False, False, False, False],
)

display(candidate_df.head(50))
candidate_df.to_csv(OUTPUT_DIR / "input_candidate_audit.csv", index=False)

def choose(backbone, manual):
    if manual is not None:
        if "smoke" in manual.lower():
            raise RuntimeError(
                f"{backbone}: manual path points to a smoke result. "
                "Use the full 60-task task-level result file."
            )
        print(f"{backbone}: manual source -> {manual}")
        return manual

    x = candidate_df[
        (candidate_df["path_hint"] == backbone)
        & (candidate_df["full_protocol_ok"])
    ]

    if x.empty:
        print(f"\nNo full 60-task candidate auto-detected for {backbone}.")
        print("Candidates for this backbone:")
        display(
            candidate_df[candidate_df["path_hint"] == backbone][
                ["spec", "n_rows", "n_tasks", "n_expected_methods",
                 "schema_ok", "full_protocol_ok", "is_smoke"]
            ]
        )
        raise RuntimeError(
            f"Cannot find a valid full 60-task result for {backbone}. "
            "Do not use smoke_task_results.csv. "
            "Add the full task-level result file or set its exact path in Cell 2."
        )

    spec = x.iloc[0]["spec"]
    print(f"{backbone}: selected full protocol source -> {spec}")
    return spec

dino_spec = choose("DINOv2-Small", DINO_RESULT_PATH)
resnet_spec = choose("ResNet-50", RESNET_RESULT_PATH)

if dino_spec == resnet_spec:
    raise RuntimeError("Both backbones resolved to the same source.")

print("\nDINO:", dino_spec)
print("ResNet:", resnet_spec)

dino = normalize_result(
    read_spec(dino_spec),
    "DINOv2-Small",
    source_spec=dino_spec,
)
resnet = normalize_result(
    read_spec(resnet_spec),
    "ResNet-50",
    source_spec=resnet_spec,
)

print("\nNormalized DINO columns:", list(dino.columns))
print("Normalized ResNet columns:", list(resnet.columns))


## 2. Frozen protocol audit

The notebook stops before inferential statistics when:
- the 60-task count is not satisfied,
- a frozen method is missing from any task,
- duplicate task-method rows exist,
- a required metric is non-finite,
- or the task grids do not align across backbones.


In [ ]:
# Cell 5 — Protocol audit
KEYS = ["dataset", "shot", "seed"]

def audit(df, backbone):
    sub = df[df.method.isin(EXPECTED_METHODS)].copy()
    methods = set(sub.method.unique())
    nt = sub[KEYS].drop_duplicates().shape[0]
    dup = int(sub.duplicated(KEYS + ["method"]).sum())
    incomplete = int((sub.groupby(KEYS).method.nunique() != len(EXPECTED_METHODS)).sum())
    nonfinite = {
        m: int((~np.isfinite(sub[m].to_numpy(float))).sum())
        for m in PRIMARY_METRICS
    }
    return {
        "backbone": backbone,
        "n_rows": len(sub),
        "n_unique_tasks": nt,
        "duplicate_key_rows": dup,
        "incomplete_tasks": incomplete,
        "missing_methods": "; ".join(sorted(set(EXPECTED_METHODS)-methods)),
        **{f"nonfinite_{m}": v for m, v in nonfinite.items()},
        "task_count_pass": nt == EXPECTED_TASKS_PER_BACKBONE,
        "method_completeness_pass": incomplete == 0 and set(EXPECTED_METHODS).issubset(methods),
        "duplicate_pass": dup == 0,
        "finite_metrics_pass": all(v == 0 for v in nonfinite.values()),
    }

dino = dino[dino.method.isin(EXPECTED_METHODS)].copy()
resnet = resnet[resnet.method.isin(EXPECTED_METHODS)].copy()

audit_df = pd.DataFrame([
    audit(dino, "DINOv2-Small"),
    audit(resnet, "ResNet-50"),
])

dino_tasks = set(map(tuple, dino[KEYS].drop_duplicates().to_numpy()))
res_tasks = set(map(tuple, resnet[KEYS].drop_duplicates().to_numpy()))
aligned = dino_tasks == res_tasks
audit_df["cross_backbone_task_alignment_pass"] = aligned

pass_cols = [c for c in audit_df if c.endswith("_pass")]
overall_protocol_pass = bool(audit_df[pass_cols].to_numpy(bool).all())

display(audit_df)
print("OVERALL PROTOCOL PASS:", overall_protocol_pass)
audit_df.to_csv(OUTPUT_DIR / "protocol_audit.csv", index=False)

if not overall_protocol_pass:
    print("DINO-only tasks:", sorted(dino_tasks-res_tasks)[:20])
    print("ResNet-only tasks:", sorted(res_tasks-dino_tasks)[:20])
    raise RuntimeError("Protocol audit failed. Inspect protocol_audit.csv.")


In [ ]:
# Cell 6 — Combine normalized frozen results
all_results = pd.concat([dino, resnet], ignore_index=True)
front = ["backbone", "dataset", "shot", "seed", "method"]
metrics_available = [m for m in ["macro_f1", "nll", "ece", "brier"] if m in all_results]
other = [c for c in all_results if c not in front + metrics_available]
all_results = all_results[front + metrics_available + other].sort_values(front).reset_index(drop=True)

print("Rows:", len(all_results))
print("Metrics:", metrics_available)
display(all_results.head(12))
all_results.to_csv(OUTPUT_DIR / "normalized_all_results.csv", index=False)


In [ ]:
# Cell 6A — Test-only provenance audit (v3)
assert all(f"test_{m}" in all_results for m in ["macro_f1","nll","ece","brier"])
provenance = []
for (backbone, method), group in all_results.groupby(["backbone", "method"]):
    for m in ["macro_f1", "nll", "ece", "brier"]:
        assert np.allclose(group[m], group[f"test_{m}"], rtol=0, atol=0), f"TEST mismatch: {backbone}, {method}, {m}"
        provenance.append({
            "backbone": backbone, "method": method, "metric": m,
            "validation_mean": float(group[f"validation_{m}"].mean()),
            "test_mean": float(group[m].mean()),
            "n_tasks": len(group)
        })
provenance_df = pd.DataFrame(provenance)
provenance_df.to_csv(OUTPUT_DIR / "audit_validation_vs_test_mean.csv", index=False)
print("Verified explicit TEST columns for all 480 method-task rows.")
display(provenance_df.head(12))


## 3. Descriptive statistics (`mean ± std`)

In [ ]:
# Cell 7 — Descriptive tables
def describe(df, groups, metrics):
    rows = []
    for keys, g in df.groupby(groups, sort=False):
        if not isinstance(keys, tuple): keys = (keys,)
        row = dict(zip(groups, keys))
        row["n"] = len(g)
        for m in metrics:
            v = g[m].dropna().to_numpy(float)
            mu = float(v.mean())
            sd = float(v.std(ddof=1)) if len(v) > 1 else np.nan
            row[f"{m}_mean"] = mu
            row[f"{m}_std"] = sd
            row[f"{m}_mean_std"] = f"{mu:.4f} ± {sd:.4f}" if len(v) > 1 else f"{mu:.4f}"
        rows.append(row)
    return pd.DataFrame(rows)

overall_desc = describe(all_results, ["backbone", "method"], metrics_available)
strat_desc = describe(all_results, ["backbone", "dataset", "shot", "method"], metrics_available)

display(overall_desc)
display(strat_desc.head(24))

overall_desc.to_csv(OUTPUT_DIR / "table_descriptive_overall.csv", index=False)
strat_desc.to_csv(OUTPUT_DIR / "table_descriptive_by_dataset_shot.csv", index=False)


## 4. Paired inference

For each frozen contrast, the notebook reports the raw metric difference and a **favorable effect**. Positive favorable effects always indicate improvement:

- Macro-F1: challenger − reference
- NLL/ECE/Brier: reference − challenger

Inference uses paired task-level bootstrap CIs and paired sign-flip permutation tests.


In [ ]:
# Cell 8 — Statistical helpers
def paired_vectors(df, backbone, reference, challenger, metric):
    sub = df[
        (df.backbone == backbone) &
        (df.method.isin([reference, challenger]))
    ][KEYS + ["method", metric]]
    wide = sub.pivot(index=KEYS, columns="method", values=metric).reset_index()
    wide = wide.dropna(subset=[reference, challenger]).copy()
    wide["raw_delta"] = wide[challenger] - wide[reference]
    wide["favorable_delta"] = (
        wide["raw_delta"] if METRIC_DIRECTIONS[metric] == "higher"
        else -wide["raw_delta"]
    )
    return wide

def bootstrap_ci(delta, n_boot=BOOTSTRAP_REPS, seed=RANDOM_SEED):
    d = np.asarray(delta, float)
    r = np.random.default_rng(seed)
    means = np.empty(n_boot)
    pos, chunk = 0, 2000
    while pos < n_boot:
        b = min(chunk, n_boot-pos)
        idx = r.integers(0, len(d), size=(b, len(d)))
        means[pos:pos+b] = d[idx].mean(axis=1)
        pos += b
    lo, hi = np.quantile(means, [ALPHA/2, 1-ALPHA/2])
    return float(lo), float(hi)

def signflip_p(delta, n_perm=PERMUTATION_REPS, seed=RANDOM_SEED):
    d = np.asarray(delta, float)
    obs = abs(d.mean())
    r = np.random.default_rng(seed)
    extreme, done, chunk = 0, 0, 5000
    while done < n_perm:
        b = min(chunk, n_perm-done)
        signs = r.choice(np.array([-1.0, 1.0]), size=(b, len(d)))
        pm = np.abs((signs*d).mean(axis=1))
        extreme += int((pm >= obs-1e-15).sum())
        done += b
    return (extreme+1)/(n_perm+1)

def holm_adjust(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    sp = p[order]
    m = len(sp)
    adj_sorted = np.empty(m)
    running = 0.0
    for i, v in enumerate(sp):
        running = max(running, (m-i)*v)
        adj_sorted[i] = min(running, 1.0)
    adj = np.empty(m)
    adj[order] = adj_sorted
    return adj

print("Statistical helpers ready.")


In [ ]:
# Cell 9 — Primary paired effects, CIs, permutation tests, Holm correction
backbones = ["DINOv2-Small", "ResNet-50"]
infer_metrics = [m for m in PRIMARY_METRICS + OPTIONAL_METRICS if m in all_results]

rows = []
for bi, b in enumerate(backbones):
    for ci, (cn, (ref, chal)) in enumerate(PRIMARY_CONTRASTS.items()):
        for mi, m in enumerate(infer_metrics):
            w = paired_vectors(all_results, b, ref, chal, m)
            d = w.favorable_delta.to_numpy(float)
            lo, hi = bootstrap_ci(d, seed=RANDOM_SEED + 1000*bi + 100*ci + 10*mi)
            p = signflip_p(d, seed=RANDOM_SEED + 5000 + 1000*bi + 100*ci + 10*mi)
            rows.append({
                "backbone": b,
                "contrast": cn,
                "reference": ref,
                "challenger": chal,
                "metric": m,
                "direction": METRIC_DIRECTIONS[m],
                "n_pairs": len(w),
                "reference_mean": float(w[ref].mean()),
                "challenger_mean": float(w[chal].mean()),
                "raw_delta_mean": float(w.raw_delta.mean()),
                "favorable_effect_mean": float(d.mean()),
                "favorable_effect_median": float(np.median(d)),
                "ci95_low": lo,
                "ci95_high": hi,
                "p_permutation_raw": p,
            })

paired_effects = pd.DataFrame(rows)

# Holm correction is restricted to the frozen confirmatory family:
# R2/R3 × Macro-F1/NLL/ECE × two backbones = 12 tests.
confirm_mask = (
    paired_effects["contrast"].isin(HOLM_CONFIRMATORY_CONTRASTS)
    & paired_effects["metric"].isin(PRIMARY_METRICS)
)
paired_effects["p_holm_confirmatory"] = np.nan
paired_effects.loc[confirm_mask, "p_holm_confirmatory"] = holm_adjust(
    paired_effects.loc[confirm_mask, "p_permutation_raw"].to_numpy(float)
)
paired_effects["significant_holm_0.05"] = (
    paired_effects["p_holm_confirmatory"] < ALPHA
)

display(paired_effects)
paired_effects.to_csv(OUTPUT_DIR / "table_paired_effects.csv", index=False)
paired_effects.loc[
    confirm_mask,
    ["backbone","contrast","metric","n_pairs","favorable_effect_mean",
     "ci95_low","ci95_high","p_permutation_raw","p_holm_confirmatory",
     "significant_holm_0.05"]
].to_csv(OUTPUT_DIR / "table_holm_tests.csv", index=False)


In [ ]:
# Cell 10 — Win / tie / loss
rows = []
for b in backbones:
    for cn, (ref, chal) in PRIMARY_CONTRASTS.items():
        for m in infer_metrics:
            d = paired_vectors(all_results, b, ref, chal, m).favorable_delta.to_numpy(float)
            wins = int((d > TIE_TOL).sum())
            ties = int((np.abs(d) <= TIE_TOL).sum())
            losses = int((d < -TIE_TOL).sum())
            rows.append({
                "backbone": b, "contrast": cn, "metric": m, "n_pairs": len(d),
                "wins": wins, "ties": ties, "losses": losses,
                "win_rate": wins/len(d),
            })
wtl = pd.DataFrame(rows)
display(wtl)
wtl.to_csv(OUTPUT_DIR / "table_wins_ties_losses.csv", index=False)


## 5. Cross-backbone consistency

In [ ]:
# Cell 11 — Cross-backbone consistency
rows = []
for cn, (ref, chal) in PRIMARY_CONTRASTS.items():
    for m in infer_metrics:
        d1 = paired_vectors(all_results, "DINOv2-Small", ref, chal, m)[KEYS+["favorable_delta"]].rename(columns={"favorable_delta":"delta_dino"})
        d2 = paired_vectors(all_results, "ResNet-50", ref, chal, m)[KEYS+["favorable_delta"]].rename(columns={"favorable_delta":"delta_resnet"})
        z = d1.merge(d2, on=KEYS, validate="one_to_one")
        x, y = z.delta_dino.to_numpy(float), z.delta_resnet.to_numpy(float)
        mask = (np.abs(x)>TIE_TOL) | (np.abs(y)>TIE_TOL)
        agree = float((np.sign(x[mask]) == np.sign(y[mask])).mean()) if mask.any() else np.nan
        rho, rho_p = spearmanr(x, y)

        e1 = paired_effects[(paired_effects.backbone=="DINOv2-Small") & (paired_effects.contrast==cn) & (paired_effects.metric==m)].iloc[0]
        e2 = paired_effects[(paired_effects.backbone=="ResNet-50") & (paired_effects.contrast==cn) & (paired_effects.metric==m)].iloc[0]

        rows.append({
            "contrast": cn, "metric": m, "n_matched_tasks": len(z),
            "mean_effect_dino": float(x.mean()),
            "mean_effect_resnet": float(y.mean()),
            "same_favorable_mean_direction": bool(e1.favorable_effect_mean>0 and e2.favorable_effect_mean>0),
            "both_ci95_exclude_zero_favorably": bool(e1.ci95_low>0 and e2.ci95_low>0),
            "task_level_sign_agreement": agree,
            "spearman_rho": float(rho),
            "spearman_p_raw_descriptive": float(rho_p),
        })
cross_backbone = pd.DataFrame(rows)
display(cross_backbone)
cross_backbone.to_csv(OUTPUT_DIR / "table_cross_backbone_consistency.csv", index=False)


## 6. Figures

In [ ]:
# Cell 12 — Figure helpers
def savefig(fig, stem):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR/f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR/f"{stem}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
# Cell 13 — Figure 1: Overall Macro-F1 with bootstrap 95% CI
rows = []
for bi, b in enumerate(backbones):
    for mi, method in enumerate(METHOD_ORDER):
        v = all_results[(all_results.backbone==b) & (all_results.method==method)].macro_f1.to_numpy(float)
        lo, hi = bootstrap_ci(v, seed=RANDOM_SEED+10000+100*bi+mi)
        rows.append({"backbone":b,"method":method,"mean":v.mean(),"lo":lo,"hi":hi})
p = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10,5.8))
x = np.arange(len(METHOD_ORDER))
offset = {"DINOv2-Small":-0.12, "ResNet-50":0.12}
for b in backbones:
    s = p[p.backbone==b].set_index("method").loc[METHOD_ORDER]
    mu = s["mean"].to_numpy()
    yerr = np.vstack([mu-s.lo.to_numpy(), s.hi.to_numpy()-mu])
    ax.errorbar(x+offset[b], mu, yerr=yerr, marker="o", linestyle="none", capsize=4, label=b)
ax.set_xticks(x)
ax.set_xticklabels(METHOD_ORDER, rotation=15, ha="right")
ax.set_ylabel("Macro-F1")
ax.set_title("Overall Macro-F1 with 95% bootstrap confidence intervals")
ax.grid(axis="y", alpha=0.3)
ax.legend()
savefig(fig, "fig01_overall_macro_f1")


In [ ]:
# Cell 14 — Figures 2–3: F1–ECE trade-off
for no, b in enumerate(backbones, start=2):
    s = all_results[all_results.backbone==b].groupby("method", as_index=False)[["macro_f1","ece"]].mean()
    fig, ax = plt.subplots(figsize=(7.5,6))
    for _, r in s.iterrows():
        ax.scatter(r.macro_f1, r.ece, s=90)
        ax.annotate(r.method, (r.macro_f1,r.ece), xytext=(6,5), textcoords="offset points")
    ax.set_xlabel("Mean Macro-F1 ↑")
    ax.set_ylabel("Mean ECE ↓")
    ax.set_title(f"F1–ECE trade-off: {b}")
    ax.grid(alpha=0.3)
    stem = "DINOv2_Small" if b=="DINOv2-Small" else "ResNet50"
    savefig(fig, f"fig{no:02d}_f1_ece_tradeoff_{stem}")


In [ ]:
# Cell 15 — Figure 4: Forest plot of favorable effects
f = paired_effects[paired_effects.metric.isin(PRIMARY_METRICS)].copy()
f["contrast"] = pd.Categorical(f.contrast, list(PRIMARY_CONTRASTS), ordered=True)
f["metric"] = pd.Categorical(f.metric, PRIMARY_METRICS, ordered=True)
f = f.sort_values(["backbone","contrast","metric"]).reset_index(drop=True)

labels = [f"{r.backbone} | {r.contrast} | {r.metric}" for r in f.itertuples()]
y = np.arange(len(f))
mu = f.favorable_effect_mean.to_numpy()
xerr = np.vstack([mu-f.ci95_low.to_numpy(), f.ci95_high.to_numpy()-mu])

fig, ax = plt.subplots(figsize=(10, max(7,0.42*len(f))))
ax.errorbar(mu,y,xerr=xerr,fmt="o",capsize=3,linestyle="none")
ax.axvline(0, linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(labels)
ax.invert_yaxis()
ax.set_xlabel("Favorable effect (positive = improvement)")
ax.set_title("Primary paired effects with 95% bootstrap confidence intervals")
ax.grid(axis="x", alpha=0.3)
savefig(fig, "fig04_primary_effects_forest")


In [ ]:
# Cell 16 — Figures 5–6: Cross-backbone ΔMacro-F1 consistency for R2 and R3
for no, cn in zip([5,6], ["R2_NormalizedBudget","R3_CalibrationAwareMO"]):
    ref, chal = PRIMARY_CONTRASTS[cn]
    x = paired_vectors(all_results,"DINOv2-Small",ref,chal,"macro_f1")[KEYS+["favorable_delta"]].rename(columns={"favorable_delta":"delta_dino"})
    y = paired_vectors(all_results,"ResNet-50",ref,chal,"macro_f1")[KEYS+["favorable_delta"]].rename(columns={"favorable_delta":"delta_resnet"})
    z = x.merge(y,on=KEYS,validate="one_to_one")
    rho,_ = spearmanr(z.delta_dino,z.delta_resnet)

    fig,ax = plt.subplots(figsize=(7,6.5))
    ax.scatter(z.delta_dino,z.delta_resnet,alpha=0.8)
    ax.axhline(0,linewidth=1); ax.axvline(0,linewidth=1)
    lo = min(z.delta_dino.min(),z.delta_resnet.min())
    hi = max(z.delta_dino.max(),z.delta_resnet.max())
    ax.plot([lo,hi],[lo,hi],linestyle="--",linewidth=1)
    ax.set_xlabel("DINOv2-Small favorable ΔMacro-F1")
    ax.set_ylabel("ResNet-50 favorable ΔMacro-F1")
    ax.set_title(f"{cn}: cross-backbone consistency\nSpearman ρ = {rho:.3f}")
    ax.grid(alpha=0.3)
    savefig(fig, f"fig{no:02d}_cross_backbone_{cn}_delta_f1")


## 7. Claim-oriented replication audit and export

In [ ]:
# Cell 17 — Replication audit
rows = []
for cn in PRIMARY_CONTRASTS:
    for m in infer_metrics:
        s = paired_effects[(paired_effects.contrast==cn) & (paired_effects.metric==m)]
        rows.append({
            "contrast": cn,
            "metric": m,
            "n_backbones": s.backbone.nunique(),
            "favorable_mean_both_backbones": bool(len(s)==2 and (s.favorable_effect_mean>0).all()),
            "ci95_positive_both_backbones": bool(len(s)==2 and (s.ci95_low>0).all()),
            "in_holm_confirmatory_family": bool(cn in HOLM_CONFIRMATORY_CONTRASTS and m in PRIMARY_METRICS),
            "holm_significant_both_backbones": bool(
                cn in HOLM_CONFIRMATORY_CONTRASTS and m in PRIMARY_METRICS
                and len(s)==2 and s["significant_holm_0.05"].all()
            ),
        })
claim_audit = pd.DataFrame(rows)
display(claim_audit)
claim_audit.to_csv(OUTPUT_DIR/"claim_replication_audit.csv",index=False)


In [ ]:
# Cell 18 — Human-readable audit report
parts = [
    "GAPFA Notebook 12 — Statistical Analysis Audit",
    "="*60,
    f"Random seed: {RANDOM_SEED}",
    f"Bootstrap repetitions: {BOOTSTRAP_REPS}",
    f"Permutation repetitions: {PERMUTATION_REPS}",
    "Holm scope: R2/R3 × Macro-F1/NLL/ECE × 2 backbones (12 hypotheses)",
    "",
    "INPUT SOURCES",
    f"DINOv2-Small: {dino_spec}",
    f"ResNet-50: {resnet_spec}",
    "",
    "PROTOCOL",
    audit_df.to_string(index=False),
    "",
    f"OVERALL PROTOCOL PASS: {overall_protocol_pass}",
    "",
    "PRIMARY EFFECTS",
    paired_effects[
        ["backbone","contrast","metric","n_pairs","favorable_effect_mean",
         "ci95_low","ci95_high","p_permutation_raw","p_holm_confirmatory","significant_holm_0.05"]
    ].to_string(index=False),
    "",
    "CROSS-BACKBONE CONSISTENCY",
    cross_backbone.to_string(index=False),
]
report = "\n".join(parts)
print(report[:12000])
(OUTPUT_DIR/"report_protocol_and_statistics.txt").write_text(report,encoding="utf-8")


In [ ]:
# Cell 19 — Manifest and ZIP bundle
manifest = {
    "notebook": "GAPFA_12_v3_Test_Corrected_Statistical_Analysis",
    "metric_source": "Explicit held-out test_* columns",
    "random_seed": RANDOM_SEED,
    "expected_tasks_per_backbone": EXPECTED_TASKS_PER_BACKBONE,
    "expected_methods": EXPECTED_METHODS,
    "primary_contrasts": {
        k: {"reference":v[0], "challenger":v[1]}
        for k,v in PRIMARY_CONTRASTS.items()
    },
    "primary_metrics": PRIMARY_METRICS,
    "holm_confirmatory_contrasts": HOLM_CONFIRMATORY_CONTRASTS,
    "optional_metrics_used": [m for m in OPTIONAL_METRICS if m in all_results],
    "bootstrap_repetitions": BOOTSTRAP_REPS,
    "permutation_repetitions": PERMUTATION_REPS,
    "alpha": ALPHA,
    "holm_scope": "R2_R3_x_macro_f1_nll_ece_x_2_backbones_12_hypotheses",
    "input_sources": {"DINOv2-Small":dino_spec, "ResNet-50":resnet_spec},
    "overall_protocol_pass": overall_protocol_pass,
}
with open(OUTPUT_DIR/"results_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2,ensure_ascii=False)

zip_path = Path("/kaggle/working/GAPFA12_v3_TEST_Statistical_Analysis_and_Figures.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p,arcname=p.relative_to(OUTPUT_DIR))

print("Output directory:", OUTPUT_DIR)
print("ZIP:", zip_path)
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file():
        print(" -",p.name)


## Expected outputs

```text
/kaggle/working/
├── GAPFA12_outputs/
│   ├── input_candidate_audit.csv
│   ├── protocol_audit.csv
│   ├── normalized_all_results.csv
│   ├── table_descriptive_overall.csv
│   ├── table_descriptive_by_dataset_shot.csv
│   ├── table_paired_effects.csv
│   ├── table_holm_tests.csv
│   ├── table_wins_ties_losses.csv
│   ├── table_cross_backbone_consistency.csv
│   ├── claim_replication_audit.csv
│   ├── report_protocol_and_statistics.txt
│   ├── results_manifest.json
│   ├── fig01_overall_macro_f1.png / .pdf
│   ├── fig02_f1_ece_tradeoff_DINOv2_Small.png / .pdf
│   ├── fig03_f1_ece_tradeoff_ResNet50.png / .pdf
│   ├── fig04_primary_effects_forest.png / .pdf
│   ├── fig05_cross_backbone_R2_NormalizedBudget_delta_f1.png / .pdf
│   └── fig06_cross_backbone_R3_CalibrationAwareMO_delta_f1.png / .pdf
└── GAPFA12_Statistical_Analysis_and_Figures.zip
```
